# Player Archetype Analysis; Feature Preprocessing (Transformation & Scaling) [Milestone 3.1]

**Project Objective**:
Gather football player and team performance data across Europe's top 5 leagues to create a scouting dashboard that leverages unsupervised Machine Learning to identify statistical profiles ("player archetypes") and uncover comparable or undervalued talents.

### Milestone 3.1 Objectives:
1. Separate descriptive columns (`player_identifier`, `Player`, `Nation`, `Age`, `Pos`, `Primary_Pos`, `Squad`, `Comp`, league playing percentages) from the numeric performance feature matrix.
2. Address heavy skewed and zero-inflated distributions via Yeo-Johnson power transformations and justified (via domain knowledge) custom weighting.
3. Standardise/scale features using `StandardScaler` and evaluate against `RobustScaler`.

## Setup

In [ ]:
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import PowerTransformer, RobustScaler, StandardScaler

# Add repo root to locate data directory
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

PRIMARY_POS_ORDER = ['GK', 'DF', 'MF', 'FW']

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

### Ingesting the Cleaned Featureset
Import `player_featureset.csv`, which has previously been processed by:
- Goalkeepers filtered out.
- Min 600 minutes played per club filter.
- Multi-club players aggregated.
- Metrics normalised to per-90 (with a `_p90` suffix).
- Redundant within-group features pruned.

In [ ]:
featureset_path = repo_root / 'data' / 'processed' / 'player_featureset.csv'
df = pd.read_csv(featureset_path)

print(f"{df.shape[0]} players, {df.shape[1]} total columns")
print(f"Missing values: {df.isna().sum().sum()}")
df.sample(10)

## Identify Modelling Features

The below features are purely for post-modelling analysis and dashboard filtering:
   - Player identity: `player_identifier`, `Player`, `Nation`, `Age`.
   - Positional assignments: `Pos`, `Primary_Pos`, `Num_Pos`.
   - Team & Competition context: `Squad`, `Comp`.
   - Categorical / engineered preferences: `passing_foot_pref`, `signed_preference_score`.
   - League minutes share: `Premier League_pct_mins`, `Ligue 1_pct_mins`, `Bundesliga_pct_mins`, `Serie A_pct_mins`, `La Liga_pct_mins`.

The rest are the performance metrics to be passed into the model.

In [ ]:
LEAGUE_PCT_MINS_COLS = [
    'Premier League_pct_mins',
    'Ligue 1_pct_mins',
    'Bundesliga_pct_mins',
    'Serie A_pct_mins',
    'La Liga_pct_mins',
]

DESCRIPTIVE_COLS = [
    'player_identifier',
    'Player',
    'Nation',
    'Age',
    'Pos',
    'Primary_Pos',
    'Num_Pos',
    'Squad',
    'Comp',
    'passing_foot_pref',
    'signed_preference_score',
    *LEAGUE_PCT_MINS_COLS,
]


def separate_features_and_metadata(
    df: pd.DataFrame,
    metadata_cols: list[str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Separates metadata/descriptive columns from numeric performance features."""
    if metadata_cols is None:
        metadata_cols = DESCRIPTIVE_COLS

    present_meta = [col for col in metadata_cols if col in df.columns]
    feature_cols = [col for col in df.columns if col not in present_meta]

    metadata_df = df[present_meta].copy()
    X = df[feature_cols].copy()

    # Ensure all remaining features are numeric
    non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric:
        raise ValueError(f"Non-numeric columns found in feature matrix: {non_numeric}")

    return metadata_df, X


metadata_df, X_raw = separate_features_and_metadata(df, metadata_cols=DESCRIPTIVE_COLS)

print(f"Metadata Matrix Shape:\n{metadata_df.shape}")
print(f"Numeric Feature Matrix Shape:\n{X_raw.shape}")

# Ensure only numeric cols are present
non_numeric_cols = X_raw.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_cols:
    raise ValueError(f"Non-numeric columns in feature matrix: {non_numeric_cols}")

# Preview metadata columns
print("\nDescriptive metadata columns:")
print(metadata_df.columns.tolist())
metadata_df.head(3)

## 3. Skewness Diagnosis & Distribution Analysis

EDA revealed that many of these metrics are heavily right skewed, and sometimes zero-inflated. For example, strikers score the majority of goals, while defenders make the majority of clearances, and only a small number of players regularly take set pieces. Other events are generally rare, such as red cards.

These heavily skewed distributions can dominate the variance in dimension reduction and the distances used to calculate clusters.

We can quantify skew (using the Fisher-Pearson skewness coefficient) to classify the level of skew across each feature. High skew may require transforming.

In [ ]:
skewness_series = X_raw.skew()
abs_skewness = skewness_series.abs()

skew_summary = pd.DataFrame({
    'Metric': skewness_series.index,
    'Skewness': skewness_series.values,
    'Abs_Skewness': abs_skewness.values,
}).sort_values('Abs_Skewness', ascending=False)

symmetric_count = (abs_skewness <= 0.5).sum()
moderate_count = ((abs_skewness > 0.5) & (abs_skewness <= 1.0)).sum()
high_count = ((abs_skewness > 1.0) & (abs_skewness <= 2.0)).sum()
severe_count = (abs_skewness > 2.0).sum()

print("=== Feature Skewness Classification ===")
print(f"Approximately Symmetric (|skew| <= 0.5):        {symmetric_count:2d} features ({symmetric_count/len(X_raw.columns)*100:5.1f}%)")
print(f"Moderately Skewed        (0.5 < |skew| <= 1.0): {moderate_count:2d} features ({moderate_count/len(X_raw.columns)*100:5.1f}%)")
print(f"Highly Skewed            (1.0 < |skew| <= 2.0): {high_count:2d} features ({high_count/len(X_raw.columns)*100:5.1f}%)")
print(f"Severely Skewed / Rare   (|skew| > 2.0):        {severe_count:2d} features ({severe_count/len(X_raw.columns)*100:5.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution of absolute skewness
sns.histplot(abs_skewness, bins=25, kde=True, color='teal', ax=axes[0])
axes[0].axvline(0.5, color='#4C72B0', linestyle='--', linewidth=1.5, label='Symmetric boundary (0.5)')
axes[0].axvline(1.0, color='#DD8452', linestyle='--', linewidth=1.5, label='High skew threshold (1.0)')
axes[0].axvline(2.0, color='#C44E52', linestyle='--', linewidth=1.5, label='Severe skew threshold (2.0)')
axes[0].set_title('Distribution of Feature Absolute Skewness Across 109 Metrics', fontsize=12, fontweight='bold')
axes[0].set_xlabel('|Skewness|')
axes[0].set_ylabel('Feature Count')
axes[0].legend()

# Top 15 most skewed features
top_15_skewed = skew_summary.head(15)
sns.barplot(data=top_15_skewed, x='Abs_Skewness', y='Metric', hue='Metric', palette='viridis', legend=False, ax=axes[1])
axes[1].set_title('Top 15 Most Skewed Performance Features', fontsize=12, fontweight='bold')
axes[1].set_xlabel('|Skewness|')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()


46 (41.8%) of the features are heavily skewed and will require transformation, with the majority of these being low count `_p90` metrics

## 4. Transformations

There are many ways to transform the data, but three of them are:
- **Box-Cox**; Requires strictly positive values, which cannot handle the fairly common 0 values in this data.
- **Log(x + 1)**: Unsuitable due to the fractional p90 values in the data, which will be unevenly compressed by the transformation.
- **Yeo-Johnson**: Supports <=0 values (such as `forward_pressing_index` and `forward_touch_index`), so is the most appropriate. It also has the benefit of being flexible, calculating a $\lambda$ value for each feature via Maximum Likelihood Estimation, which in turn applies a different transformation depending on the shape & skew of the data.

The latter will therefore be applied to all features exceeding the $|skew| > 1.0$ threshold.

In [ ]:
def detect_skewed_features(
    X: pd.DataFrame,
    threshold: float = 1.0,
) -> list[str]:
    """Identifies features whose absolute skewness exceeds a given threshold."""
    skews = X.skew().abs()
    return skews[skews > threshold].index.tolist()


def power_transform_features(
    X: pd.DataFrame,
    columns: list[str] | None = None,
    method: str = 'yeo-johnson',
    skew_threshold: float = 1.0,
) -> tuple[pd.DataFrame, PowerTransformer]:
    """Applies power transformation to reduce skewness and stabilise variance."""
    if columns is None:
        columns = detect_skewed_features(X, threshold=skew_threshold)

    X_trans = X.copy()
    if not columns:
        transformer = PowerTransformer(method=method, standardize=False)
        return X_trans, transformer

    transformer = PowerTransformer(method=method, standardize=False)
    transformed_vals = transformer.fit_transform(X[columns])
    X_trans[columns] = transformed_vals

    return X_trans, transformer


skewed_cols = detect_skewed_features(X_raw, threshold=1.0)
print(f"Applying Yeo-Johnson transformation to {len(skewed_cols)} skewed features...")

X_transformed, pt = power_transform_features(X_raw, columns=skewed_cols, method='yeo-johnson')

skew_comparison = pd.DataFrame({
    'Metric': skewed_cols,
    'Raw_Skew': X_raw[skewed_cols].skew().values,
    'Transformed_Skew': X_transformed[skewed_cols].skew().values,
    'Fitted_Lambda': pt.lambdas_,
})
skew_comparison['Abs_Reduction'] = skew_comparison['Raw_Skew'].abs() - skew_comparison['Transformed_Skew'].abs()
skew_comparison = skew_comparison.sort_values(by='Raw_Skew', key=abs, ascending=False)

print(f"Mean absolute skewness across transformed features:")
print(f"  Raw:         {X_raw[skewed_cols].skew().abs().mean():.3f}")
print(f"  Transformed: {X_transformed[skewed_cols].skew().abs().mean():.3f}")

skew_comparison.head(15)

In [ ]:
skewness_series = X_transformed.skew()
abs_skewness = skewness_series.abs()

skew_summary = pd.DataFrame({
    'Metric': skewness_series.index,
    'Skewness': skewness_series.values,
    'Abs_Skewness': abs_skewness.values,
}).sort_values('Abs_Skewness', ascending=False)

symmetric_count = (abs_skewness <= 0.5).sum()
moderate_count = ((abs_skewness > 0.5) & (abs_skewness <= 1.0)).sum()
high_count = ((abs_skewness > 1.0) & (abs_skewness <= 2.0)).sum()
severe_count = (abs_skewness > 2.0).sum()

still_skewed_features = list(skew_summary[skew_summary.Abs_Skewness > 1].Metric)

print("=== Feature Skewness Classification ===")
print(f"Approximately Symmetric (|skew| <= 0.5):        {symmetric_count:2d} features ({symmetric_count/len(X_transformed.columns)*100:5.1f}%)")
print(f"Moderately Skewed        (0.5 < |skew| <= 1.0): {moderate_count:2d} features ({moderate_count/len(X_transformed.columns)*100:5.1f}%)")
print(f"Highly Skewed            (1.0 < |skew| <= 2.0): {high_count:2d} features ({high_count/len(X_transformed.columns)*100:5.1f}%)")
print(f"Severely Skewed / Rare   (|skew| > 2.0):        {severe_count:2d} features ({severe_count/len(X_transformed.columns)*100:5.1f}%)")

print(f"\nStill Skewed Features  (1.0 < |skew|): [{len(still_skewed_features)} features]\n        {still_skewed_features}")


In [ ]:
# Compare distribution before and after transformation
fig, axes = plt.subplots(len(still_skewed_features), 2, figsize=(14, 3 * len(still_skewed_features)))

for i, col in enumerate(still_skewed_features):
    raw_s = X_raw[col].skew()
    trans_s = X_transformed[col].skew()

    sns.histplot(X_raw[col], kde=True, color='#4C72B0', ax=axes[i, 0], stat='density', bins=30)
    axes[i, 0].set_title(f"Raw: {col} (Skew: {raw_s:.2f})", fontweight='bold', fontsize=11)
    axes[i, 0].set_xlabel('')

    sns.histplot(X_transformed[col], kde=True, color='#55A868', ax=axes[i, 1], stat='density', bins=30)
    axes[i, 1].set_title(f"Yeo-Johnson: {col} (Skew: {trans_s:.2f})", fontweight='bold', fontsize=11)
    axes[i, 1].set_xlabel('')

plt.tight_layout()
plt.show()


Even after transformation, 13 columns are still heavily skewed by zero-inflation in metrics relating to uncommon actions. If left as-is, these values are likely to dominate any dimension reduction and clustering.

There are several strategies we can use to address this:
- Weighting can be applied to dampen the possible spread of a high-variance metric, such as errors, penalties conceded, or blocked shots.
- For very rare metrics (e.g. own goals), we may want to remove these; with the justification that a season or two of data is unlikely to be a high enough sample (compared to the very low counts) to determine a repeated pattern vs random events.
- Depending on the metric meaning, we could categorise into True/False (1/0) features, representing whether the player performed that action **at all** (e.g. are they a penalty taker)
- Aggregate related, low-count features (such as yellow and red cards).

It is also worth noting that many of these may only be zero-inflated across all positions; e.g. BlkShSv_p90 (number of goal-bound shots blocked per 90) is likely to be far more common among defenders than all positions. This means a stratified, per-position pipeline may well handle these much better. However, with the issue of limited positional data (already highlighted in the EDA), this will be left for a v2 of this project. This is to prevent a player who plays multiple position from skewing the data, by bleeding across positions (e.g. a player who plays as a defender as well as a forward is likely to have much higher attacking metrics than one who has only played as a defender).

---

Of these remaining features, two are related to red cards (`2CrdY_p90` and `CrdR_p90`). A weighted sum of these and yellow cards - weighted using the UEFA Fair Play calculation weighting (3 points for a red, 1 for a yellow) - to account for the increased impact of a red versus a yellow card) will reduce this zero-inflation, as yellow cards are far more common.

As discussed in the EDA, `ShoPK_p90` and `ShoFK_p90` ony give an indication of whether the player is trusted to take these set pieces, rather than how successful they are from them. These can instead be converted to binary faetures, representing attacking set piece takers (after first assigning a lower boundary).

Own goals are very rare, and are unlikely to be statistically significant over just one season of data; these will therefore be dropped from the model.

Err_p90 and PKcon_p90 are both key defensive statistics, however do not truely reflect the player's 'style', and is as much of a product of the team around them as their archetype. As they are heavily zero-inflated, they will be dampened as above, Again though, these are likely to be a prime contender to be included in a future defenders-only stratification of the  model.

GcaDrib, GcaFld, and GcaDef, GcaPassDead_p90 are all related to goals created from dribbles, being fouled, defensive actions, and set piece passes, respectively. However, their signal is already largely captured by the 'Sca' equivalents, which relate to shots created instead, as well as the broader GCA_p90. This can therefore be dropped, with the Sca equivalents retained, with the further justification that the conversion of a shot created to a goal is not necessarily reflective of the player themselves. These again will likely be more effective in the future stratified model.

Similarly, BlkShSv_p90 is largely covered by BlkSh_p90, Blocks_p90, and Clr_p9, and is primarily isolated to defenders. PKwon_p90 will be largely covered by CPA_p90, TouAttPen_p90, and ScaFld_p90. and is primarily isolated to forwards. These will therefore be dropped for now, but re-explored in the stratified model.

